# 06 — Model Catalog
## Decision Science Laboratory inside `bd_replica_crm`

Catálogo académico + selector + laboratorio ejecutable para:

**OLS / Logit / Poisson / Survival / Propensity / IPW / DiD / RDD / Synthetic Control / Uplift**

Cada método incluye:
- pregunta que responde;
- supuesto central;
- tablas/campos candidatos;
- readiness;
- ejemplo ejecutable o plantilla;
- gate para evitar conclusiones inválidas.

> Modelo estadístico no equivale automáticamente a identificación causal.


In [1]:
from __future__ import annotations
import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

settings=load_settings(PROJECT_ROOT)
conn=connect_postgres(settings)

pd.set_option("display.max_columns",180)
pd.set_option("display.max_rows",180)
pd.set_option("display.width",260)

def sql_df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)

def exists(schema,name):
    q=sql_df("""
    SELECT EXISTS(
      SELECT 1 FROM information_schema.tables
      WHERE table_schema=%s AND table_name=%s
      UNION ALL
      SELECT 1 FROM information_schema.views
      WHERE table_schema=%s AND table_name=%s
    ) ok
    """,[schema,name,schema,name])
    return bool(q.iloc[0]["ok"])

print("DB:",settings.postgres.database)


DB: medallio_dw


## 1. Catálogo maestro


In [2]:
catalog=pd.DataFrame([
["OLS","¿Cuánto cambia Y cuando cambia X?","Outcome continuo / interpretación","Exogeneidad condicional","lead_evidence / pricing / tiempos"],
["Logit","¿Cómo cambia la probabilidad de un evento?","Separación / minuta","Especificación + independencia condicional","lead_evidence"],
["Poisson","¿Qué determina el número de eventos?","Conteos","Media condicional bien especificada","interacciones / procesos / agregados"],
["Survival","¿Qué acelera o retrasa un evento?","Tiempo hasta separación/minuta","Censura no informativa; PH si Cox","lead_evidence + ciclo comercial"],
["Propensity","¿Qué probabilidad había de recibir tratamiento?","Comparabilidad observacional","Ignorabilidad + overlap","recommendations/actions + features"],
["IPW","¿Cuál sería el ATE reponderando por tratamiento?","ATE observacional","Propensity correcto + positivity","actions/outcomes/features"],
["DiD","¿Qué cambió después del tratamiento vs control?","Políticas en el tiempo","Parallel trends + no anticipación","proyecto/asesor × tiempo"],
["RDD","¿Qué efecto tiene cruzar un umbral?","Reglas por cutoff","Continuidad + no manipulación","priority_score + action"],
["Synthetic Control","¿Qué habría pasado sin intervención?","Unidad tratada vs donor pool","Buen pre-fit + donor pool válido","proyectos × tiempo"],
["Uplift","¿Para quién funciona más la acción?","Tratamiento heterogéneo","Comparabilidad + overlap","actions/outcomes/features"],
],columns=["model","question","best_for","core_assumption","bd_replica_use"])
catalog


,model,question,best_for,core_assumption,bd_replica_use
0,OLS,¿Cuánto cambia Y cuando cambia X?,Outcome continuo / interpretación,Exogeneidad condicional,lead_evidence / pricing / tiempos
1,Logit,¿Cómo cambia la probabilidad de un evento?,Separación / minuta,Especificación + independencia condicional,lead_evidence
2,Poisson,¿Qué determina el número de eventos?,Conteos,Media condicional bien especificada,interacciones / procesos / agregados
3,Survival,¿Qué acelera o retrasa un evento?,Tiempo hasta separación/minuta,Censura no informativa; PH si Cox,lead_evidence + ciclo comercial
4,Propensity,¿Qué probabilidad había de recibir tratamiento?,Comparabilidad observacional,Ignorabilidad + overlap,recommendations/actions + features
5,IPW,¿Cuál sería el ATE reponderando por tratamiento?,ATE observacional,Propensity correcto + positivity,actions/outcomes/features
6,DiD,¿Qué cambió después del tratamiento vs control?,Políticas en el tiempo,Parallel trends + no anticipación,proyecto/asesor × tiempo
7,RDD,¿Qué efecto tiene cruzar un umbral?,Reglas por cutoff,Continuidad + no manipulación,priority_score + action
8,Synthetic Control,¿Qué habría pasado sin intervención?,Unidad tratada vs donor pool,Buen pre-fit + donor pool válido,proyectos × tiempo
9,Uplift,¿Para quién funciona más la acción?,Tratamiento heterogéneo,Comparabilidad + overlap,actions/outcomes/features


## 2. Selector


In [3]:
MODEL="Logit"
selected=catalog[catalog["model"].eq(MODEL)]
display(selected.T if len(selected) else selected)


,1
model,Logit
question,¿Cómo cambia la probabilidad de un evento?
best_for,Separación / minuta
core_assumption,Especificación + independencia condicional
bd_replica_use,lead_evidence


## 3. Readiness de objetos


In [4]:
objects=[
("features","lead_evidence"),
("decision_intelligence","lead_scores"),
("decision_intelligence","recommendations"),
("decision_intelligence","actions"),
("decision_intelligence","outcomes"),
("core","fact_ciclo_comercial_unidad"),
]
readiness=pd.DataFrame([(s,o,exists(s,o)) for s,o in objects],columns=["schema","object","exists"])
readiness


C:\Users\dinat\AppData\Local\Temp\ipykernel_29144\3924303252.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,schema,object,exists
0,features,lead_evidence,True
1,decision_intelligence,lead_scores,True
2,decision_intelligence,recommendations,True
3,decision_intelligence,actions,True
4,decision_intelligence,outcomes,True
5,core,fact_ciclo_comercial_unidad,True


## 4. Dataset común


In [5]:
lead=sql_df("""
SELECT
 evidence_key,lead_id,decision_at,codigo_proyecto,asesor,canal,medio,
 hour_of_day,day_of_week,is_weekend,
 client_prior_assignments_90d,days_since_previous_assignment,
 project_leads_90d,project_sep_rate_90d,project_minuta_rate_180d,
 advisor_leads_90d,advisor_sep_rate_90d,advisor_minuta_rate_180d,
 global_sep_rate_90d,global_minuta_rate_180d,
 separacion_14d,minuta_60d
FROM features.lead_evidence
""")
print("rows:",len(lead))
lead.head()


C:\Users\dinat\AppData\Local\Temp\ipykernel_29144\3924303252.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


rows: 206043


,evidence_key,lead_id,decision_at,codigo_proyecto,asesor,canal,medio,hour_of_day,day_of_week,is_weekend,client_prior_assignments_90d,days_since_previous_assignment,project_leads_90d,project_sep_rate_90d,project_minuta_rate_180d,advisor_leads_90d,advisor_sep_rate_90d,advisor_minuta_rate_180d,global_sep_rate_90d,global_minuta_rate_180d,separacion_14d,minuta_60d
0,f994e4aa385df9f82790804055fd4556,122839,2024-07-31 15:27:45.230310+00:00,GY,None,None,3 días de locura inmobiliaria,10,3,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
1,16503383413d5776f724ccd18ceeb415,122840,2024-07-17 17:04:44.809686+00:00,GY,None,None,3 días de locura inmobiliaria,12,3,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
2,e1682806e4b5bef9ad763360af5b0e05,122845,2024-07-09 20:59:49.638926+00:00,EEUU,None,None,3 días de locura inmobiliaria,15,2,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
3,f6a2ac8b5f26fb4c966d0727fd7c6b91,122846,2024-07-30 02:30:44.796950+00:00,FX,None,None,3 días de locura inmobiliaria,21,1,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
4,c51a3999e74e7d2ff62053ec8b8bea96,122848,2024-07-09 20:59:49.638926+00:00,FX,None,None,3 días de locura inmobiliaria,15,2,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0


## 5. OLS


In [6]:
ols_data=lead[[
"minuta_60d","project_sep_rate_90d","advisor_sep_rate_90d",
"client_prior_assignments_90d","is_weekend"
]].dropna()

if len(ols_data)>=100:
    ols=smf.ols(
        "minuta_60d ~ project_sep_rate_90d + advisor_sep_rate_90d + client_prior_assignments_90d + is_weekend",
        data=ols_data
    ).fit(cov_type="HC3")
    print(ols.summary())
else:
    print("PENDING")


PENDING


## 6. Logit + efectos marginales


In [7]:
logit_data=lead[[
"separacion_14d","project_sep_rate_90d","advisor_sep_rate_90d",
"client_prior_assignments_90d","is_weekend"
]].dropna()

if len(logit_data)>=100 and logit_data["separacion_14d"].nunique()==2:
    logit=smf.logit(
        "separacion_14d ~ project_sep_rate_90d + advisor_sep_rate_90d + client_prior_assignments_90d + is_weekend",
        data=logit_data
    ).fit(disp=False)
    print(logit.summary())
    print(logit.get_margeff().summary())
else:
    print("PENDING")


PENDING


## 7. Poisson


In [8]:
poisson_df=(
    lead.assign(decision_date=pd.to_datetime(lead["decision_at"],utc=True).dt.date)
    .groupby(["decision_date","codigo_proyecto"],dropna=False)
    .agg(leads=("evidence_key","count"),
         avg_project_sep_rate=("project_sep_rate_90d","mean"))
    .reset_index()
)

if len(poisson_df)>=30:
    poisson=smf.glm(
        "leads ~ avg_project_sep_rate",
        data=poisson_df,
        family=sm.families.Poisson()
    ).fit()
    print(poisson.summary())
else:
    print("PENDING")


ValueError: negative dimensions are not allowed

## 8. Survival


In [ ]:
if exists("core","fact_ciclo_comercial_unidad"):
    survival=sql_df("""
    SELECT
      e.evidence_key,e.decision_at,c.fecha_separacion,
      e.project_sep_rate_90d,e.advisor_sep_rate_90d
    FROM features.lead_evidence e
    LEFT JOIN core.fact_ciclo_comercial_unidad c
      ON c.documento_cliente=e.documento_cliente
     AND COALESCE(c.codigo_proyecto_ciclo,c.codigo_proyecto_unidad)=e.codigo_proyecto
     AND c.fecha_separacion>=e.decision_at::date
    """)
    survival["decision_at"]=pd.to_datetime(survival["decision_at"],utc=True,errors="coerce")
    survival["fecha_separacion"]=pd.to_datetime(survival["fecha_separacion"],utc=True,errors="coerce")
    now=pd.Timestamp.now(tz="UTC")
    survival["event"]=survival["fecha_separacion"].notna().astype(int)
    end=survival["fecha_separacion"].fillna(now)
    survival["duration_days"]=(end-survival["decision_at"]).dt.total_seconds()/86400
    surv_model=survival[
        ["duration_days","event","project_sep_rate_90d","advisor_sep_rate_90d"]
    ].dropna()
    surv_model=surv_model[surv_model["duration_days"]>=0]
    print("rows:",len(surv_model),"events:",surv_model["event"].sum())
    if len(surv_model)>100 and surv_model["event"].sum()>20:
        exog=sm.add_constant(surv_model[["project_sep_rate_90d","advisor_sep_rate_90d"]])
        try:
            ph=sm.duration.PHReg(
                surv_model["duration_days"],exog,status=surv_model["event"]
            ).fit()
            print(ph.summary())
        except Exception as exc:
            print("PHReg:",repr(exc))
else:
    print("PENDING")


## 9. Dataset treatment/control


In [ ]:
if exists("decision_intelligence","recommendations") and exists("decision_intelligence","actions"):
    treatment=sql_df("""
    SELECT
      r.recommendation_id,
      r.entity_id AS lead_id,
      r.created_at AS recommendation_at,
      a.action_taken,a.action_at,a.action_cost,
      le.project_sep_rate_90d,
      le.advisor_sep_rate_90d,
      le.global_sep_rate_90d,
      le.client_prior_assignments_90d,
      le.separacion_14d,
      le.minuta_60d
    FROM decision_intelligence.recommendations r
    LEFT JOIN decision_intelligence.actions a
      ON a.recommendation_id=r.recommendation_id
    LEFT JOIN features.lead_evidence le
      ON le.lead_id=r.entity_id
    WHERE r.decision_system='priorizacion_leads'
    """)
    treatment["treated"]=treatment["action_taken"].notna().astype(int)
else:
    treatment=pd.DataFrame()

print("rows:",len(treatment))


## 10. Propensity


In [ ]:
ps_cols=[
"project_sep_rate_90d","advisor_sep_rate_90d",
"global_sep_rate_90d","client_prior_assignments_90d"
]

if len(treatment):
    ps_data=treatment[["treated","minuta_60d"]+ps_cols].dropna().copy()
    if len(ps_data)>50 and ps_data["treated"].nunique()==2:
        X=ps_data[ps_cols]
        t=ps_data["treated"]
        ps_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(X,t)
        ps_data["propensity"]=ps_model.predict_proba(X)[:,1]
        display(ps_data.groupby("treated")["propensity"].describe())
    else:
        print("PENDING")
else:
    ps_data=pd.DataFrame()
    print("PENDING")


## 11. IPW


In [ ]:
if len(ps_data) and "propensity" in ps_data.columns:
    p=ps_data["propensity"].clip(.01,.99)
    t=ps_data["treated"]
    y=ps_data["minuta_60d"]
    w=t/p+(1-t)/(1-p)
    mu1=np.average(y[t==1],weights=w[t==1])
    mu0=np.average(y[t==0],weights=w[t==0])
    print("IPW ATE:",mu1-mu0)
else:
    print("PENDING")


## 12. DiD


In [ ]:
DID_TREATED_PROJECT=None
DID_CONTROL_PROJECT=None
DID_INTERVENTION_DATE=None

if DID_TREATED_PROJECT and DID_CONTROL_PROJECT and DID_INTERVENTION_DATE:
    did=(lead.assign(date=pd.to_datetime(lead["decision_at"],utc=True).dt.date)
         .groupby(["date","codigo_proyecto"],as_index=False)
         .agg(outcome=("separacion_14d","mean")))
    did=did[did["codigo_proyecto"].isin([DID_TREATED_PROJECT,DID_CONTROL_PROJECT])].copy()
    did["treated"]=did["codigo_proyecto"].eq(DID_TREATED_PROJECT).astype(int)
    did["post"]=(pd.to_datetime(did["date"])>=pd.Timestamp(DID_INTERVENTION_DATE)).astype(int)
    if len(did)>20:
        did_model=smf.ols("outcome ~ treated + post + treated:post",data=did).fit(cov_type="HC3")
        print(did_model.summary())
else:
    print("PENDING: define treated/control/date")


## 13. RDD


In [ ]:
RDD_CUTOFF=70.0

if exists("decision_intelligence","lead_scores"):
    rdd=sql_df("""
    SELECT s.priority_score,le.minuta_60d AS outcome
    FROM decision_intelligence.lead_scores s
    JOIN features.lead_evidence le USING (evidence_key)
    WHERE le.minuta_60d IS NOT NULL
    """)
    if len(rdd)>100:
        bw=10
        x=rdd[rdd["priority_score"].between(RDD_CUTOFF-bw,RDD_CUTOFF+bw)].copy()
        x["running"]=x["priority_score"]-RDD_CUTOFF
        x["above"]=(x["running"]>=0).astype(int)
        if len(x)>50 and x["above"].nunique()==2:
            rdd_model=smf.ols("outcome ~ running + above + running:above",data=x).fit(cov_type="HC3")
            print(rdd_model.summary())
        else:
            print("PENDING")
    else:
        print("PENDING")
else:
    print("PENDING")


## 14. Synthetic Control


In [ ]:
SYNTH_TREATED_PROJECT=None
SYNTH_INTERVENTION_DATE=None

panel=(lead.assign(date=pd.to_datetime(lead["decision_at"],utc=True).dt.date)
       .groupby(["date","codigo_proyecto"],as_index=False)
       .agg(outcome=("separacion_14d","mean")))

if SYNTH_TREATED_PROJECT and SYNTH_INTERVENTION_DATE:
    pivot=panel.pivot(index="date",columns="codigo_proyecto",values="outcome").sort_index()
    intervention=pd.Timestamp(SYNTH_INTERVENTION_DATE).date()
    pre=pivot.loc[pivot.index<intervention].copy()
    if SYNTH_TREATED_PROJECT in pivot.columns:
        donors=[c for c in pivot.columns if c!=SYNTH_TREATED_PROJECT]
        train=pre[[SYNTH_TREATED_PROJECT]+donors].dropna()
        if len(train)>10 and donors:
            y=train[SYNTH_TREATED_PROJECT].values
            X=train[donors].values
            model=Ridge(alpha=1.0,fit_intercept=False,positive=True).fit(X,y)
            weights=pd.Series(model.coef_,index=donors)
            if weights.sum():
                weights=weights/weights.sum()
            display(weights.sort_values(ascending=False).head(20).to_frame("weight"))
else:
    print("PENDING: define treated project/date")


## 15. Uplift


In [ ]:
if len(treatment):
    uplift_data=treatment[["treated","minuta_60d"]+ps_cols].dropna().copy()
    if len(uplift_data)>100 and uplift_data["treated"].nunique()==2 and uplift_data["minuta_60d"].nunique()==2:
        X=uplift_data[ps_cols]
        t=uplift_data["treated"]
        y=uplift_data["minuta_60d"]
        mt=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
        mc=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
        mt.fit(X[t==1],y[t==1])
        mc.fit(X[t==0],y[t==0])
        uplift_data["uplift_hat"]=mt.predict_proba(X)[:,1]-mc.predict_proba(X)[:,1]
        display(uplift_data["uplift_hat"].describe(percentiles=[.1,.25,.5,.75,.9]))
    else:
        print("PENDING")
else:
    print("PENDING")


## 16. Readiness por modelo


In [ ]:
ready=[]

def add(model,status,reason):
    ready.append({"model":model,"status":status,"reason":reason})

add("OLS","READY" if len(ols_data)>=100 else "PENDING",f"n={len(ols_data):,}")
add("Logit","READY" if len(logit_data)>=100 and logit_data["separacion_14d"].nunique()==2 else "PENDING",f"n={len(logit_data):,}")
add("Poisson","READY" if len(poisson_df)>=30 else "PENDING",f"n={len(poisson_df):,}")
add("Survival","READY" if "surv_model" in globals() and len(surv_model)>100 else "PENDING",f"n={len(surv_model) if 'surv_model' in globals() else 0:,}")
add("Propensity","READY" if len(ps_data)>50 and "propensity" in ps_data.columns else "PENDING",f"n={len(ps_data):,}")
add("IPW","READY" if len(ps_data) and "propensity" in ps_data.columns else "PENDING","requiere propensity")
add("DiD","READY" if DID_TREATED_PROJECT and DID_CONTROL_PROJECT and DID_INTERVENTION_DATE else "PENDING","requiere treated/control/date")
add("RDD","READY" if exists("decision_intelligence","lead_scores") else "PENDING","requiere scores + cutoff")
add("Synthetic Control","READY" if SYNTH_TREATED_PROJECT and SYNTH_INTERVENTION_DATE else "PENDING","requiere treated/date")
add("Uplift","READY" if len(treatment)>100 and treatment.get("treated",pd.Series()).nunique()==2 else "PENDING",f"n={len(treatment):,}")

model_readiness=pd.DataFrame(ready)
model_readiness


## 17. Pregunta de negocio → método


In [ ]:
decision_map=pd.DataFrame([
["¿Qué variables se asocian con mayor conversión?","OLS / Logit"],
["¿Qué aumenta la probabilidad de separación?","Logit"],
["¿Qué determina el número de eventos?","Poisson"],
["¿Qué acelera el tiempo hasta separación/minuta?","Survival"],
["¿La acción comercial causó un cambio?","RCT / IPW / DiD / RDD según diseño"],
["¿Qué ocurrió después de una política en un proyecto?","DiD"],
["¿Qué efecto tuvo cruzar un cutoff?","RDD"],
["¿Qué habría pasado sin intervención?","Synthetic Control"],
["¿A quién conviene intervenir porque responde más?","Uplift"],
],columns=["business_question","recommended_model"])
decision_map


## 18. Escalera de evidencia

```text
Descriptivo
→ Predictivo
→ Asociación ajustada
→ Causal
→ Heterogeneidad causal
```

La herramienta estadística no reemplaza el diseño de identificación.


## 19. Próximos modelos


In [ ]:
future_models=pd.DataFrame([
["Negative Binomial","conteos con sobredispersión"],
["Panel Fixed Effects","heterogeneidad no observada"],
["Event Study","parallel trends y dinámica temporal"],
["Interrupted Time Series","intervención temporal"],
["IV / 2SLS","endogeneidad con instrumento"],
["Double Machine Learning","causalidad con alta dimensión"],
["Causal Forest","heterogeneous treatment effects"],
["Bayesian Hierarchical","shrinkage por proyecto/asesor"],
],columns=["model","use"])
future_models


In [ ]:
conn.close(); print("Conexión cerrada.")
